# Generate Official GLUE Submission Files

This notebook creates official GLUE submission files from our fine-tuned checkpoints.

For an official submission on GLUE, all 9 task and an addtional diagnotic task has to be tested. 


In [1]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer

print('Imports successful')

Imports successful


In [ ]:
# Config
BASE_DIR = Path.cwd()
CONFIG_PATH = BASE_DIR / 'configs' / 'submission_config.json'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

def to_abs_path(maybe_path):
    if maybe_path is None:
        return None
    p = Path(maybe_path)
    return (BASE_DIR / p).resolve() if not p.is_absolute() else p

MODEL_PATH = to_abs_path(cfg.get('model_path'))
TOKENIZER_PATH = to_abs_path(cfg.get('tokenizer_path'))
BASE_MODEL_NAME = cfg.get('base_model_name')

# Optional per-task overrides (recommended for full GLUE submissions)
TASK_MODEL_PATHS = {k.lower(): to_abs_path(v) for k, v in cfg.get('task_model_paths', {}).items()}
TASK_TOKENIZER_PATHS = {k.lower(): to_abs_path(v) for k, v in cfg.get('task_tokenizer_paths', {}).items()}
TASK_BASE_MODELS = {k.lower(): v for k, v in cfg.get('task_base_model_names', {}).items()}

TASKS_TO_SUBMIT = [t.lower() for t in cfg['tasks_to_submit']]
MAX_LENGTH = int(cfg.get('max_length', 512))
BATCH_SIZE = int(cfg.get('batch_size', 64))
SUBMISSION_NAME = cfg.get('submission_name', 'glue_submission')
OUTPUT_ROOT = (BASE_DIR / cfg.get('output_dir', './outputs')).resolve()

print('Loaded configuration:')
print(f'  Default model path: {MODEL_PATH}')
print(f'  Default tokenizer path: {TOKENIZER_PATH}')
print(f'  Default base model name: {BASE_MODEL_NAME}')
print(f'  Per-task model paths: {sorted(TASK_MODEL_PATHS.keys())}')
print(f'  Per-task tokenizer paths: {sorted(TASK_TOKENIZER_PATHS.keys())}')
print(f'  Per-task base model names: {sorted(TASK_BASE_MODELS.keys())}')
print(f'  Tasks: {TASKS_TO_SUBMIT}')
print(f'  Max length: {MAX_LENGTH}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Output root: {OUTPUT_ROOT}')

In [ ]:
# Task registry
# label_mode:
#   - 'int': classification id
#   - 'float': regression value
#   - 'string_from_feature': map predicted class id to dataset label string
#   - 'nli_string_from_model': map predicted class id via model.config.id2label
TASK_SPECS = {
    'sst2': {
        'input_columns': ['sentence'],
        'test_splits': ['test'],
        'output_files': ['SST-2.tsv'],
        'label_mode': 'int',
    },
    'cola': {
        'input_columns': ['sentence'],
        'test_splits': ['test'],
        'output_files': ['CoLA.tsv'],
        'label_mode': 'int',
    },
    'mrpc': {
        'input_columns': ['sentence1', 'sentence2'],
        'test_splits': ['test'],
        'output_files': ['MRPC.tsv'],
        'label_mode': 'int',
    },
    'qqp': {
        'input_columns': ['question1', 'question2'],
        'test_splits': ['test'],
        'output_files': ['QQP.tsv'],
        'label_mode': 'int',
    },
    'stsb': {
        'input_columns': ['sentence1', 'sentence2'],
        'test_splits': ['test'],
        'output_files': ['STS-B.tsv'],
        'label_mode': 'float',
    },
    'mnli': {
        'input_columns': ['premise', 'hypothesis'],
        'test_splits': ['test_matched', 'test_mismatched'],
        'output_files': ['MNLI-m.tsv', 'MNLI-mm.tsv'],
        'label_mode': 'string_from_feature',
    },
    'qnli': {
        'input_columns': ['question', 'sentence'],
        'test_splits': ['test'],
        'output_files': ['QNLI.tsv'],
        'label_mode': 'string_from_feature',
    },
    'rte': {
        'input_columns': ['sentence1', 'sentence2'],
        'test_splits': ['test'],
        'output_files': ['RTE.tsv'],
        'label_mode': 'string_from_feature',
    },
    'wnli': {
        'input_columns': ['sentence1', 'sentence2'],
        'test_splits': ['test'],
        'output_files': ['WNLI.tsv'],
        'label_mode': 'int',
    },
    'ax': {
        'input_columns': ['premise', 'hypothesis'],
        'test_splits': ['test'],
        'output_files': ['AX.tsv'],
        'label_mode': 'nli_string_from_model',
    },
}

unknown = [t for t in TASKS_TO_SUBMIT if t not in TASK_SPECS and t != 'ax']
if unknown:
    raise ValueError(f'Unsupported task(s): {unknown}. Supported tasks: {sorted(TASK_SPECS)}')

print('Task registry ready')
print('AX special case (official): use MNLI-trained model to infer AX and write AX.tsv')

Task registry ready


## AX Special Case (Official GLUE Path)

For submitting to the GLUE Benchmark site, an additional dataset (the "diagnostics main" ("ax") dataset) has to be included. This was not mentioned in the paper. 

This notebook enforces the official diagnostic flow for AX task:
1. Use an MNLI-trained checkpoint.
2. Infer on GLUE AX.
3. Write AX.tsv for submission.

By default, AX uses task_model_paths['mnli'] if available.

In [ ]:
def tokenize_split(split_ds, tokenizer, input_columns, max_length):
    def _tokenize(batch):
        if len(input_columns) == 1:
            return tokenizer(batch[input_columns[0]], truncation=True, max_length=max_length)
        return tokenizer(batch[input_columns[0]], batch[input_columns[1]], truncation=True, max_length=max_length)

    return split_ds.map(_tokenize, batched=True, remove_columns=split_ds.column_names)


def postprocess_predictions(task_name, split_ds, logits, label_mode, model=None):
    if label_mode == 'float':
        preds = np.asarray(logits).squeeze()
        # STS-B is bounded between 0 and 5
        return np.clip(preds, 0.0, 5.0).astype(float).tolist()

    class_ids = np.argmax(logits, axis=1)

    if label_mode == 'int':
        return class_ids.astype(int).tolist()

    if label_mode == 'string_from_feature':
        label_feature = split_ds.features.get('label', None)
        if label_feature is None or not hasattr(label_feature, 'names') or label_feature.names is None:
            raise ValueError(f'Cannot map labels to strings for task {task_name}; missing ClassLabel names.')
        return [label_feature.names[int(i)] for i in class_ids]

    if label_mode == 'nli_string_from_model':
        if model is None or not hasattr(model, 'config') or not hasattr(model.config, 'id2label'):
            raise ValueError(f'Cannot map NLI labels for task {task_name}; model id2label is missing.')

        # Preferred source for canonical order if available in this dataset split.
        split_label_names = None
        label_feature = split_ds.features.get('label', None)
        if label_feature is not None and hasattr(label_feature, 'names') and label_feature.names is not None:
            split_label_names = list(label_feature.names)

        default_nli_order = ['entailment', 'neutral', 'contradiction']
        mapped = []
        for i in class_ids:
            raw = str(model.config.id2label[int(i)]).strip().lower()
            if 'entail' in raw:
                mapped.append('entailment')
            elif 'contrad' in raw:
                mapped.append('contradiction')
            elif 'neutral' in raw:
                mapped.append('neutral')
            elif raw.startswith('label_') and raw.split('_')[-1].isdigit():
                idx = int(raw.split('_')[-1])
                if split_label_names is not None and idx < len(split_label_names):
                    mapped.append(str(split_label_names[idx]))
                elif idx < len(default_nli_order):
                    mapped.append(default_nli_order[idx])
                else:
                    raise ValueError(f'Unexpected generic NLI label index: {raw}')
            else:
                raise ValueError(f'Unexpected NLI label value in model config: {raw}')
        return mapped

    raise ValueError(f'Unknown label mode: {label_mode}')


def write_glue_tsv(idx_values, predictions, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame({'index': idx_values, 'prediction': predictions})
    df.to_csv(output_path, sep='\t', index=False)


def get_idx_column(split_ds):
    if 'idx' in split_ds.column_names:
        return split_ds['idx']
    if 'index' in split_ds.column_names:
        return split_ds['index']
    return list(range(len(split_ds)))

In [ ]:
def resolve_model_path(path_obj: Path) -> Path:
    if path_obj is None:
        raise ValueError('Model path is None. Set model_path or task_model_paths in config.')

    if not path_obj.exists():
        raise FileNotFoundError(f'Model path not found: {path_obj}')

    if (path_obj / 'config.json').exists():
        return path_obj

    checkpoint_dirs = sorted(
        [p for p in path_obj.glob('checkpoint-*') if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
    )
    if checkpoint_dirs:
        chosen = checkpoint_dirs[-1]
        print(f'Model path has no config.json; using latest checkpoint: {chosen}')
        return chosen

    return path_obj

def load_tokenizer_for_task(task_name: str, resolved_model_path: Path):
    tokenizer_load_errors = []
    tokenizer = None

    task_tokenizer_path = TASK_TOKENIZER_PATHS.get(task_name)
    task_base_model = TASK_BASE_MODELS.get(task_name)

    sources = [task_tokenizer_path, TOKENIZER_PATH, resolved_model_path, BASE_MODEL_NAME, task_base_model]
    for source in sources:
        if source is None:
            continue
        try:
            tokenizer = AutoTokenizer.from_pretrained(source)
            print(f'Loaded tokenizer from: {source}')
            return tokenizer
        except Exception as exc:
            tokenizer_load_errors.append((str(source), str(exc)))

    debug_msg = '\n'.join([f'  - {src}: {err}' for src, err in tokenizer_load_errors])
    raise RuntimeError(
        f'Failed to load tokenizer for task {task_name}. Set task_tokenizer_paths[{task_name}] or fallback tokenizer_path/base_model_name.\n'
        + debug_msg
    )

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
submission_dir = OUTPUT_ROOT / f'{SUBMISSION_NAME}_{timestamp}'
submission_dir.mkdir(parents=True, exist_ok=True)

generated_files = []

# Official GLUE diagnostic path: use MNLI trained model -> infer AX -> write AX.tsv
tasks_to_run = list(TASKS_TO_SUBMIT)
if 'ax' not in tasks_to_run:
    tasks_to_run.append('ax')

for task_name in tasks_to_run:
    spec = TASK_SPECS[task_name]
    dataset_name = 'ax' if task_name == 'ax' else task_name
    dataset = load_dataset('glue', dataset_name)

    # For AX, use MNLI checkpoint by default, then fallback to AX-specific or global path.
    if task_name == 'ax':
        selected_model_path = TASK_MODEL_PATHS.get('mnli', TASK_MODEL_PATHS.get('ax', MODEL_PATH))
    else:
        selected_model_path = TASK_MODEL_PATHS.get(task_name, MODEL_PATH)

    resolved_model_path = resolve_model_path(selected_model_path)

    print(f'\nTask: {task_name}')
    print(f'  Model source: {resolved_model_path}')

    tokenizer = load_tokenizer_for_task(task_name, resolved_model_path)
    model = AutoModelForSequenceClassification.from_pretrained(resolved_model_path)
    model.to(device)

    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

    for split_name, output_name in zip(spec['test_splits'], spec['output_files']):
        if split_name not in dataset:
            raise KeyError(f'Split {split_name} not found for task {task_name}. Available: {list(dataset.keys())}')

        split_ds = dataset[split_name]
        idx_values = get_idx_column(split_ds)

        encoded = tokenize_split(
            split_ds=split_ds,
            tokenizer=tokenizer,
            input_columns=spec['input_columns'],
            max_length=MAX_LENGTH,
        )

        pred_output = trainer.predict(encoded)
        predictions = postprocess_predictions(
            task_name=task_name,
            split_ds=split_ds,
            logits=pred_output.predictions,
            label_mode=spec['label_mode'],
            model=model,
        )

        out_path = submission_dir / output_name
        write_glue_tsv(idx_values, predictions, out_path)
        generated_files.append(out_path)
        print(f'  Wrote: {out_path.name} ({len(predictions)} predictions)')

zip_path = submission_dir.with_suffix('.zip')
with ZipFile(zip_path, 'w') as zf:
    for file_path in generated_files:
        zf.write(file_path, arcname=file_path.name)

print('\nDone.')
print(f'Submission folder: {submission_dir}')
print(f'Submission zip:    {zip_path}')

## Manual Next Steps

Keep task output filenames exactly as generated (e.g., `SST-2.tsv`).

The generted ZIP file is uploaded to GLUE: Keep in mind: only 2 uploads per 24 hours are allowed. 

